# 06. Two Pointers: Advanced Patterns

More problems that use the same idea in less obvious forms: **allowing duplicates**, **merging sorted arrays**, **mountains**, **counting triples**, and **O(1)-space comparisons**.

**Learning order:** REMOVE DUPLICATES II → INTERSECTION → LONGEST MOUNTAIN → VALID TRIANGLE NUMBER → PARITY II → BACKSPACE COMPARE O(1) → INTERVIEW THINKING

## 1. Remove Duplicates from Sorted Array II (allow at most 2)

General trick for "allow at most `k` copies": write `nums[fast]` if `write < k` or it **differs from `nums[write - k]`** (the element `k` places back in the **kept** part).

In [1]:
def remove_duplicates_ii(nums, k=2):
    write = 0
    for x in nums:
        if write < k or x != nums[write - k]:
            nums[write] = x
            write += 1
    return write

nums = [0, 0, 1, 1, 1, 1, 2, 3, 3]
n = remove_duplicates_ii(nums)
print(n, nums[:n])   # 7 [0, 0, 1, 1, 2, 3, 3]

7 [0, 0, 1, 1, 2, 3, 3]


## 2. Intersection of Two Arrays II (with repeats)

Sort both, then walk with two pointers: equal → take it and advance both; smaller → advance that pointer. O(n log n + m log m) time, O(1) extra space. (With a hash map it is O(n + m) time, O(min(n, m)) space.)

In [2]:
def intersect(a, b):
    a, b = sorted(a), sorted(b)
    i = j = 0
    res = []
    while i < len(a) and j < len(b):
        if a[i] == b[j]:
            res.append(a[i]); i += 1; j += 1
        elif a[i] < b[j]:
            i += 1
        else:
            j += 1
    return res

print(intersect([1, 2, 2, 1], [2, 2]), intersect([4, 9, 5], [9, 4, 9, 8, 4]))   # [2, 2] [4, 9]

[2, 2] [4, 9]


## 3. Longest Mountain in Array

A mountain strictly increases then strictly decreases (length ≥ 3). For each **peak** (`a[i-1] < a[i] > a[i+1]`), expand left and right with pointers. Each element is visited a constant number of times: O(n).

In [3]:
def longest_mountain(arr):
    best, i, n = 0, 1, len(arr)
    while i < n - 1:
        if arr[i - 1] < arr[i] > arr[i + 1]:           # a peak
            l, r = i - 1, i + 1
            while l > 0 and arr[l - 1] < arr[l]:
                l -= 1
            while r < n - 1 and arr[r] > arr[r + 1]:
                r += 1
            best = max(best, r - l + 1)
            i = r                                       # skip past this mountain
        else:
            i += 1
    return best

print(longest_mountain([2, 1, 4, 7, 3, 2, 5]), longest_mountain([2, 2, 2]))   # 5 0

5 0


## 4. Valid Triangle Number (count triples)

Sort. Fix the **largest** side at index `k`. Use `i = 0`, `j = k - 1`: if `nums[i] + nums[j] > nums[k]`, then **every** `i..j-1` pairs with `j` to form a triangle (`j - i` triples), so move `j` left; otherwise move `i` right. O(n²).

In [4]:
def triangle_number(nums):
    nums.sort()
    count = 0
    for k in range(len(nums) - 1, 1, -1):
        i, j = 0, k - 1
        while i < j:
            if nums[i] + nums[j] > nums[k]:
                count += j - i
                j -= 1
            else:
                i += 1
    return count

print(triangle_number([2, 2, 3, 4]), triangle_number([4, 2, 3, 4]))   # 3 4

3 4


## 5. Sort Array By Parity II (two pointers, two parities)

Even indexes must hold even numbers, odd indexes odd numbers. Pointer `i` walks the even indexes, `j` the odd ones. When `nums[i]` is odd (wrong) and `nums[j]` is even (wrong), swap them.

In [5]:
def sort_array_by_parity_ii(nums):
    j = 1
    for i in range(0, len(nums), 2):
        if nums[i] % 2:                                # wrong number at an even index
            while nums[j] % 2:                         # find a wrong number at an odd index
                j += 2
            nums[i], nums[j] = nums[j], nums[i]
    return nums

out = sort_array_by_parity_ii([4, 2, 5, 7])
print(all(out[i] % 2 == i % 2 for i in range(len(out))))   # True

True


## 6. Backspace String Compare in O(1) space

Walk **both strings from the end**. Skip characters cancelled by `#` (count pending backspaces). Compare the next real characters.

In [6]:
def backspace_compare(s, t):
    def next_valid(string, i):
        skip = 0
        while i >= 0:
            if string[i] == "#":
                skip += 1
            elif skip:
                skip -= 1
            else:
                return i
            i -= 1
        return -1
    i, j = len(s) - 1, len(t) - 1
    while True:
        i, j = next_valid(s, i), next_valid(t, j)
        if i < 0 or j < 0:
            return i < 0 and j < 0
        if s[i] != t[j]:
            return False
        i -= 1
        j -= 1

print(backspace_compare("ab#c", "ad#c"), backspace_compare("a##c", "#a#c"), backspace_compare("a#c", "b"))   # True True False

True True False


## 7. Interview thinking

**Signals:** "at most k duplicates", "intersection of sorted arrays", "mountain/peak", "count triples", "compare in O(1) space".

**Say:** "I sort, then fix one element and move two pointers; when the condition holds for `(i, j)` it holds for the whole range between, so I can count in bulk."

**Common mistakes:** wrong back-reference in the "at most k" trick (use `write - k`, not `read - k`), forgetting to skip past a mountain (O(n²)), not sorting before the triangle count, edge cases with only `#` strings.

## Quick Revision

- At most `k` copies: write if `write < k or x != nums[write - k]`.
- Intersection with repeats: sort + two pointers.
- Mountain: expand from each peak, jump to the end of the mountain.
- Triangles: sort, fix the largest side, bulk count `j - i`.
- O(1)-space compare: walk from the end, skip backspaces.